# 🧭 Cost-Optimized Model Routing for Financial Services 💸

Welcome to our **Model Router** tutorial! A wealth-management firm handles thousands of customer questions a day — from "What are your branch hours?" to "Analyze the tax impact of rebalancing my $2M portfolio." Sending *every* prompt to a frontier model is slow and expensive. Sending them all to a cheap model degrades quality on hard tasks.

In this notebook we'll:

1. **Initialize** an `AIProjectClient` against Microsoft Foundry.
2. **Create three agents** pinned to a cost/quality tier — `gpt-4.1-nano` → `gpt-4.1-mini` → `gpt-4o`.
3. **Route** FSI prompts of varying complexity and print which model handled each + the rationale.
4. **Explain** when a single managed **Foundry model-router deployment** is the better production choice.

### ⚠️ Financial Disclaimer ⚠️
> This notebook is for educational/demonstration purposes only and is **not** financial advice. Always consult a licensed advisor.

> 💡 Per-task routing matches model cost to task complexity — frontier prices only when the task truly needs them.


In [ ]:
# 1. Setup — load .env and initialize the AIProjectClient (Entra / az login)
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

load_dotenv()

project_endpoint = os.environ["AI_FOUNDRY_PROJECT_ENDPOINT"]
default_model = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

credential = AzureCliCredential()
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential, allow_preview=True)
openai_client = project_client.get_openai_client()

print("✅ Connected to:", project_endpoint)
print("🤖 Default model deployment:", default_model)


## 2. Define cost tiers and create a tier of agents 🪜

We map task complexity to three deployed models, cheapest first. Only models that actually exist in the project are used — missing tiers degrade gracefully to the next available one.

| Tier | Model | Best for | Relative cost |
|------|-------|----------|---------------|
| 🟢 Lite | `gpt-4.1-nano` | Greetings, FAQ, classification | ~1x |
| 🟡 Standard | `gpt-4.1-mini` | Product comparisons, summaries | ~5x |
| 🔴 Frontier | `gpt-4o` | Portfolio/tax/regulatory analysis | ~25x |

Each agent shares the same FSI persona; only the underlying model changes.


In [ ]:
# 2. Create one agent per tier (skip tiers whose model isn't deployed)
INSTRUCTIONS = (
    "You are a wealth-management assistant for a retail bank. Be concise, include a brief "
    "'not financial advice' disclaimer on substantive answers, and never invent figures."
)

TIERS = [
    {"tier": "lite",     "model": "gpt-4.1-nano", "cost": 1},
    {"tier": "standard", "model": "gpt-4.1-mini", "cost": 5},
    {"tier": "frontier", "model": "gpt-4o",        "cost": 25},
]

agents = {}
for t in TIERS:
    try:
        a = project_client.agents.create_version(
            agent_name=f"wealth-router-{t['tier']}",
            definition=PromptAgentDefinition(model=t["model"], instructions=INSTRUCTIONS),
        )
        agents[t["tier"]] = {"agent": a, **t}
        print(f"✅ {t['tier']:9s} -> {t['model']:14s} (agent {a.name} v{a.version})")
    except Exception as e:
        print(f"⚠️  {t['tier']:9s} -> {t['model']} unavailable, skipping: {e}")

if not agents:
    raise RuntimeError("No tier models deployed; cannot demonstrate routing.")


## 3. Route prompts by complexity and report the choice 🚦

A tiny client-side classifier scores each prompt (keywords + length) and picks the cheapest sufficient tier. We then run the prompt on that tier's agent and print **model + rationale** — the savings come from not paying frontier prices for simple questions.


In [ ]:
# 3. Complexity classifier + run FSI prompts of varying difficulty
def classify(prompt: str) -> tuple[str, str]:
    """Pick cheapest sufficient tier; return (tier, rationale)."""
    p = prompt.lower()
    hard = ("analyze", "tax", "rebalance", "regulatory", "compliance", "scenario", "portfolio")
    if any(k in p for k in hard) or len(prompt) > 160:
        return "frontier", "complex analysis / long context → frontier model"
    if any(k in p for k in ("compare", "difference", "explain", "vs", "summari")):
        return "standard", "moderate reasoning → standard model"
    return "lite", "simple FAQ/greeting → lite model"

def route(prompt: str):
    tier, why = classify(prompt)
    while tier not in agents:  # degrade to next available tier
        tier = {"lite": "standard", "standard": "frontier", "frontier": "lite"}[tier]
    chosen = agents[tier]
    r = openai_client.responses.create(
        input=prompt,
        extra_body={"agent_reference": {"type": "agent_reference", "name": chosen["agent"].name}},
    )
    print(f"👤 {prompt}")
    print(f"   🧭 tier={tier} ({chosen['model']}, ~{chosen['cost']}x) — {why}")
    print(f"   🤖 {(r.output_text or '').strip()[:240]}…\n")

for q in [
    "Hi, what are your branch opening hours?",
    "What's the difference between a Roth IRA and a traditional IRA?",
    "Analyze the tax impact of rebalancing my $2M portfolio across equities and bonds this quarter.",
]:
    route(q)


## 4. When to use a Foundry model-router deployment 🏭

The hand-written classifier above is great for teaching, but in production you'd deploy a **model router** model in Foundry and set it as the agent's base model. Then routing happens **per turn, server-side, with zero routing code**:

- One endpoint, one rate limit; `response.model` reveals the model actually used.
- Routing modes: **Balanced** (default), **Quality**, **Cost** — plus optional model subsets.
- Best when traffic is diverse and cost matters. Pin a single model only when you need identical behavior on every request.

Create it once (`router-balanced`/`router-cost`/`router-frontier`) and assign it as the agent's model — no per-prompt tier logic needed. Now we clean up the demo agents.


In [ ]:
# 5. Cleanup — delete the demo agents
for tier, info in agents.items():
    try:
        # [kept-in-foundry] project_client.agents.delete_version(agent_name=info["agent"].name, agent_version=info["agent"].version)
        print(f"🗑️ Deleted {info['agent'].name}")
    except Exception as e:
        print(f"⚠️ Could not delete {info['agent'].name}: {e}")
project_client.close()
print("✅ Cleanup complete")

## 📚 References

- [Model router for Microsoft Foundry](https://learn.microsoft.com/azure/foundry/openai/concepts/model-router) — concept and why to use it
- [Use model router for Microsoft Foundry](https://learn.microsoft.com/azure/foundry/openai/how-to/model-router) — deploy and monitor the router model
- [Use model router with Foundry agents](https://learn.microsoft.com/azure/foundry/openai/how-to/model-router-agents) — auto/direct routing for agents
- [Optimize model cost and performance](https://learn.microsoft.com/azure/foundry/control-plane/how-to-optimize-cost-performance) — cost-optimized routing strategy
> 💡 Tip: search the official docs live from your agent via the **Microsoft Learn MCP** at `https://learn.microsoft.com/api/mcp`.